# x2c_aptid-sojamilho — Aptidão Agrícola GAEZ v4 (Índice 0 a 10.000)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** aptidão agrícola para soja e milho (GAEZ), índice 0–10.000, municipal, estático*  
**Conceito e Mecanismo Econômico:** Índice biofísico adimensional de adequação de solo e clima para cultivo de soja e milho (0 a 10.000).  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__suitability_yield__gaez_v4_6190__candidatos.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar variantes sx, por produto e alto/baixo insumo; mesmas ressalvas de cenário e temporalidade.
> - Índice adimensional; não é rendimento em kg/ha nem receita monetária quando multiplicado pelo preço.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`variavel_raster`**: Camada de aptidão ('sxHr0_soy', 'sxHr0_mze', etc.).
- **`valor_medio`**: Índice médio municipal de aptidão (0 a 10.000).


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__suitability_yield__gaez_v4_6190__candidatos.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x2c_aptid-sojamilho`:
- **Atenção:** Explorar variantes sx, por produto e alto/baixo insumo; mesmas ressalvas de cenário e temporalidade.
- **Atenção:** Índice adimensional; não é rendimento em kg/ha nem receita monetária quando multiplicado pelo preço.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['valor_medio'] = pd.to_numeric(df['valor_medio'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})

# REGRA DO TXT: 'Índice adimensional; não é rendimento em kg/ha nem receita monetária quando multiplicado pelo preço.'
df_apt = df[df['variavel_raster'].str.startswith('sx')].copy()
df_apt = df_apt.groupby(['codigo_ibge', 'variavel_raster'], as_index=False)['valor_medio'].mean()
print(f"Municípios únicos: {df_apt['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da Aptidão GAEZ (0 a 10.000):")
print(df_apt.groupby('variavel_raster')['valor_medio'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_apt, x='variavel_raster', y='valor_medio', palette='mako')
plt.title("Aptidão Agrícola GAEZ (0 a 10.000)")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_piv = df_apt.pivot(index='codigo_ibge', columns='variavel_raster', values='valor_medio')
if 'sxHr0_soy' in df_piv.columns and 'sxHr0_mze' in df_piv.columns:
    sns.scatterplot(data=df_piv, x='sxHr0_soy', y='sxHr0_mze', color='navy')
    plt.title("Aptidão Soja vs Milho")
    plt.show()

## Fase 6: Feature Engineering — Aptidão Agrícola GAEZ v4 e Efeitos Fixos

### 1. Step 1: Normalização Padronizada (0 a 1)
- O índice bruto GAEZ varia de 0 a 10.000 (índice adimensional de aptidão agroclimática).
- Dividimos por 10.000 para criar uma escala intuitiva entre 0,0 (inapto) e 1,0 (aptidão máxima).

### 2. Step 2: Interação Shift-Share para Modelos em Painel
- **O Desafio dos Efeitos Fixos:** A aptidão do solo é biofisicamente estática. Em um painel com Efeito Fixo Municipal ($\alpha_i$), qualquer variável estática é eliminada da equação.
- **A Solução:** Multiplicamos a aptidão municipal pelo choque de preços mundiais de soja:
  $$Interacao_{i, t} = AptidaoNorm_i \times \ln(PrecoSoja_{t-1})$$
  Isso testa a hipótese central: municípios com solos mais aptos reagem mais intensamente a choques positivos de preços internacionais, convertendo mais terra nativa.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Pivotar o índice de aptidão por cultura
df_apt_piv = df.pivot_table(index='codigo_ibge', columns='variavel_raster', values='valor_medio').reset_index()

# 2. Normalização para escala 0.0 a 1.0
for col in df_apt_piv.columns:
    if col != 'codigo_ibge':
        df_apt_piv[f'{col}_norm'] = df_apt_piv[col] / 10000.0

# 3. Categorização Ordinal de Aptidão (Classes de Suporte Agronômico)
# aptidao_soja_classe: Alta (> 0.70), Média (0.40 a 0.70), Baixa (< 0.40)
if 'sxHr0_soy_norm' in df_apt_piv.columns:
    df_apt_piv['aptidao_soja_classe'] = pd.cut(
        df_apt_piv['sxHr0_soy_norm'],
        bins=[-0.1, 0.40, 0.70, 1.0],
        labels=['Baixa', 'Média', 'Alta']
    )

print("Aptidão Agroclimática GAEZ v4 Normalizada:")
print(df_apt_piv[['codigo_ibge', 'sxHr0_soy_norm', 'aptidao_soja_classe']].head())